# Emueg: A machine learning demo showing the power of emulators

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krisaalstad/emueg/blob/main/emueg.ipynb)


![Model meets emulator](https://raw.githubusercontent.com/krisaalstad/emueg/main/images/emulators.jpg)

*Ostrich: Diego Delso, CC BY-SA 4.0. Emu: JJ Harrison, CC BY-SA 4.0. Both via Wikimedia Commons; combined and labelled by K. Aalstad, CC BY-SA 4.0.*

### Emulating satellite retrievals with machine learning 

By Kristoffer Aalstad

In this example our goal is to use machine learning to rapidly emulate an expensive spectral unmixing model used to retrieve fractional snow-covered area (fSCA) from satellite (Sentinel-2) surface reflectances that is described in e.g. [Aalstad et al. (2020)](https://doi.org/10.1016/j.rse.2019.111618). This approach is being used actively to speed up the retrieval process and allow it to be used for larger areas  (e.g., [Mazzolini et al. (2025)](https://doi.org/10.5194/tc-19-3831-2025) and [Willmes et al. (2025)](https://doi.org/10.5194/egusphere-2025-3142)). As a reference, the unmixing model takes over a second to run for one pixel since it involves a costly iterative non-negative least squares regression. This idea of **model emulation** (also known as *surrogate modeling*, see [Gramacy 2020](https://bobby.gramacy.com/surrogates/surrogates.pdf)) using machine learning can be extended to mimic any geophysical model that we may have, from hydrological forecasting to numerical weather prediction and climate projections to name a few. The idea is to replace (or augment) an often slower geophysical model with a machine learning model that can emulate the same input to output mapping quite accurately but in a tiny fraction of the time.


We will use the `scikit-learn` library [Pedregosa et al. (2011](https://jmlr.org/papers/v12/pedregosa11a.html) since it is a well documented and relatively light-weight option to quickly dive into some plug-and-play machine learning, but many other great choices exist for Python especially for **deep learning** (e.g. `JAX, PyTorch`). Due to the so-called no free lunch theorem, it is typically recommended for each problem you are solving that you experiment with different model types (e.g. support vector machines, Gaussian processes, random forests, and neural networks) and different hyperparameter settings for these. 



Let's get started for this example by loading the required modules. The data has already been processed to make it easy to run this example (data wrangling is often most of the work).

Read in the required data

In [ ]:
ncf='S2_FCLSUM_MLB.nc'
fh=Dataset(ncf,mode='r')
print(fh) # Show contents (description, variables, and dims)
R = np.ma.filled(fh.variables['R'][:], np.nan).T
fSCA = np.ma.filled(fh.variables['fSCA'][:], np.nan).T
fh.close()

Inspect the data

In [ ]:
fig, axs = plt.subplots(1, 3)

true_color=(0,1,2)
axs[0].imshow(R[:,:,true_color])
#axs[0].axis('equal')
axs[0].set_title('True color', fontsize=10)


false_color=(4,3,1)
axs[1].imshow(R[:,:,false_color])
#axs[0].axis('equal')
axs[1].set_title('False color', fontsize=10)


im2=axs[2].imshow(fSCA)
im2.set_clim(0,1)
#axs[1].axis('equal')
axs[2].set_title('fSCA', fontsize=10)

fig.tight_layout()

Clean the data

In [ ]:
these=~np.isnan(fSCA)
y=fSCA[these] # target variable (predictand)
Nb=np.size(R,2) # Number of spectral bands
Np=np.size(y) # Number of pixels with data
X=np.zeros((Np,Nb))
X=R[these]
#for j in range(Nb):
#    tmp=R[:,:,j];
#    X[:,j]=tmp[these]
print(Np)

Randomly select some pixels to train on and train an ML algorithm

In [ ]:
frac=0.0025; # Fraction of pixels to train on.
Nt=int(np.round(frac*Np)) # Number of pixels to train on
rng = np.random.default_rng(42) # Seed the meaning of life
training=rng.choice(Np,Nt,replace=False) # Pick training indices at random
train = np.zeros(Np, dtype=bool) # Convert to a boolean array (train yes or no)
train[training]=True
print(Nt)

Train either a Gaussian process, support vector machine, or random forest (or something entirely different)

In [ ]:
X_train=X[train,:]
y_train=y[train]


ml_method='GP'
#ml_method='RF'
#ml_method='SVR'

if ml_method=='SVR':
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVR 
    from sklearn.svm import LinearSVR
    regr = make_pipeline(StandardScaler(), SVR(C=1, epsilon=0.01))
elif ml_method=='RF':
    from sklearn.ensemble import RandomForestRegressor
    regr = RandomForestRegressor(max_depth=10, n_estimators=400, random_state=0)
elif ml_method=='GP':
    import sklearn.gaussian_process.kernels as krn
    from sklearn.gaussian_process import GaussianProcessRegressor
    kernel = (krn.ConstantKernel(1.0)
              * krn.Matern(length_scale=np.ones(Nb), nu=1.5)    # ARD: one length scale per band
              + krn.WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e-1)))
    regr = GaussianProcessRegressor(kernel=kernel, optimizer='fmin_l_bfgs_b',
                                  n_restarts_optimizer=9, normalize_y=True)

regr.fit(X_train,y_train)



if ml_method=='GP':
    # The Gaussian process is a Bayesian method that is uncertainty-aware 
    # (knows when it does not know), so also output the posterior predictive
    # standard deviation for this. 
    y_pred,y_predstd=regr.predict(X,return_std=True)
    print(regr.kernel_)  # Optimized hyperparameters
    bands = ['R', 'G', 'B', 'NIR', 'SWIR1', 'SWIR2']
    ell = regr.kernel_.k1.k2.length_scale
    for b, l in zip(bands, ell):
        print(f'{b:>6}: length scale = {l:.3g}')
    # SWIR1 gets the shortest length scale, i.e. it is the most informative band, consistent
    # with the classic Normalized Difference Snow Index (NDSI), 
    # which combines green and SWIR1 reflectance.
else:
    y_pred=regr.predict(X)

Map the results back to an image

In [ ]:
fSCA_emulate=np.zeros_like(fSCA)
fSCA_emulate[these]=y_pred
fSCA_emulate[~these]=np.nan

Plot the absolute error of the emulator

In [ ]:
plt.figure()
abserr=np.abs(fSCA_emulate-fSCA)
cm=plt.imshow(abserr)
plt.title('Absolute error')
plt.ylabel('Row index, i')
plt.xlabel('Column index, j')
plt.colorbar()
plt.show()

For the Gaussian process, check if the uncertainty estimates are well calibrated in the sense that areas of high uncertainty correspond to areas of higher error.

In [ ]:
# Check visually if the uncertainty estimates from the GP are well calibrated 
if ml_method=='GP':
    #plt.figure()
    
    fig, axs = plt.subplots(1, 2)

    im0=axs[0].imshow(abserr)
    im0.set_clim(0,0.5)
    axs[0].set_title('Absolute error', fontsize=10)
    fig.colorbar(im0, ax=axs[0],fraction=0.05)
    stdim=np.zeros_like(fSCA)
    stdim[these]=y_predstd
    stdim[~these]=np.nan
    im1=axs[1].imshow(2*stdim)
    im1.set_clim(0,0.5)
    axs[1].set_title(r
    
    'Posterior uncertainty 2$\sigma$', fontsize=10)
    fig.colorbar(im0, ax=axs[1],fraction=0.05)

    fig.tight_layout()

Put the "true" map and the emulated map next to one another.

In [ ]:
fig, axs = plt.subplots(1, 2)

im0=axs[0].imshow(fSCA_emulate)
#axs[0].axis('equal')
im0.set_clim(0,1)
axs[0].set_title('emulated', fontsize=10)


im1=axs[1].imshow(fSCA)
im1.set_clim(0,1)
#axs[1].axis('equal')
axs[1].set_title('target', fontsize=10)

fig.tight_layout()

Perform some quantiative model evaluation and visualize the fit with a scatter plot. 

In [ ]:
sqerr = (y_pred - y)**2
rmse_train = np.sqrt(sqerr[train].mean())
rmse_test  = np.sqrt(sqerr[~train].mean())
r2 = np.corrcoef(y_pred[~train], y[~train])[0, 1]**2
titlestr = 'Train rmse=%4.2f, Test rmse=%4.2f, Test R$^2$=%4.2f' % (rmse_train, rmse_test, r2)

fig = plt.figure()
plt.hexbin(y, y_pred, bins='log')
coefs = np.polyfit(y, y_pred, 1)
xr = np.array([0, 1])
plt.plot(xr, xr, color=(0, 0, 0), alpha=0.4, label='1:1')
plt.plot(xr, coefs[0]*xr + coefs[1], color=(0.8, 0, 0), alpha=0.4, label='LBF')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel('Unmixed fSCA')
plt.ylabel('Emulated fSCA')
plt.title(titlestr)
plt.legend(loc='upper left')
plt.gca().set_aspect(1)
fig.tight_layout()